# 📘 Modèle Pyomo généré automatiquement

## 📦 Imports

In [1]:
from pyomo.environ import *
from pyomo.opt import SolverFactory
import pandas as pd

## 💾 Chargement des données
Les données `.dat` sont chargées nativement par Pyomo via `model.create_instance(...)` dans la section du modèle.

## 🔹 Model

In [2]:
from pyomo.environ import *

model = AbstractModel()

## 🔹 Sets

In [3]:
model.COMPARTIMENTS = Set()
model.CHARGES = Set()
model.ARC = Set(dimen=2, initialize=lambda m: [(i,j) for i in m.COMPARTIMENTS for j in m.CHARGES])

## 🔹 Parameters

In [4]:
model.CapPoids = Param(model.COMPARTIMENTS, within=NonNegativeReals)
model.CAP_VOLUME = Param(model.COMPARTIMENTS, within=NonNegativeReals)
model.Poids = Param(model.CHARGES, within=NonNegativeReals)
model.Volume = Param(model.CHARGES, within=NonNegativeReals)
model.GAIN = Param(model.CHARGES, within=NonNegativeReals)

## 🔹 Variables

In [5]:
model.X = Var(model.COMPARTIMENTS, model.CHARGES, domain=NonNegativeReals)

## 🔹 Data

In [6]:
model = model.create_instance('../data/Cargo_pipeline_clean_data.dat')

## 🔹 Constraints

In [7]:
model.c_for_0 = ConstraintList()
for c in model.COMPARTIMENTS:
    model.c_for_0.add(sum(model.X[c,j] for j in model.CHARGES) <= model.CapPoids[c])
model.c_for_1 = ConstraintList()
for c in model.COMPARTIMENTS:
    model.c_for_1.add(sum(model.Volume[j] * model.X[c,j] for j in model.CHARGES) <= model.CAP_VOLUME[c])
model.c_for_2 = ConstraintList()
for j in model.CHARGES:
    model.c_for_2.add(sum(model.X[c,j] for c in model.COMPARTIMENTS) <= model.Poids[j])
model.c_for_3 = ConstraintList()
for c in model.COMPARTIMENTS:
    model.c_for_3.add(sum(model.CapPoids[u] for u in model.COMPARTIMENTS) * sum(model.X[c,j] for j in model.CHARGES) == model.CapPoids[c] * sum(model.X[t,j] for t,j in model.ARC))

## 🔹 Objective

In [8]:
model.obj = Objective(expr=sum(sum(( model.GAIN[j] * model.X[c,j] ) for j in model.CHARGES) for c in model.COMPARTIMENTS), sense=maximize)

## ⚙️ Résolution du modèle

In [9]:
solver = SolverFactory('highs')
result = solver.solve(model, tee=True)

print('✅ Solver status:', result.solver.status)
print('✅ Termination condition:', result.solver.termination_condition)

✅ Solver status: ok
✅ Termination condition: optimal


## 🎯 Valeur de la fonction objective

In [10]:
for obj in model.component_objects(Objective, active=True):
    print(f'Objectif: {obj.name}')
    print(f'Valeur optimale: {obj():.4f}')
    print(f'Sens: {"Minimisation" if obj.sense == minimize else "Maximisation"}')

Objectif: obj
Valeur optimale: 13330.0000
Sens: Maximisation


## 📊 Valeurs optimales des variables

In [11]:
# Extraction des résultats dans un DataFrame
results_data = []
for v in model.component_objects(Var, active=True):
    for index in v:
        results_data.append({
            'Variable': v.name,
            'Index': str(index) if index != None else '-',
            'Valeur': v[index].value
        })

df_results = pd.DataFrame(results_data)
# Filtrer les valeurs non-nulles pour plus de clarté
df_results = df_results[df_results['Valeur'].notna()]
df_results = df_results[df_results['Valeur'] != 0]
df_results.style.format({'Valeur': '{:.4f}'}).set_caption('Variables de décision optimales')

,Variable,Index,Valeur
2,X,"('Front', 3)",11.0000
3,X,"('Front', 4)",1.0000
5,X,"('Center', 2)",6.0000
7,X,"('Center', 4)",12.0000
8,X,"('Back', 1)",10.0000
